# Project: Plan Your Trip with Kayak & Weather Data

## Objective
The goal of this project is to recommend the **top 5 cities in France** to visit and the **top 20 hotels** in that area, based on the best weather forecast for the coming days.

## Workflow
1.  **Geolocation**: Retrieve GPS coordinates (latitude, longitude) for a list of target cities using the Nominatim API.
2.  **Weather Forecast**: Fetch 5-day weather forecasts for each city using the OpenWeatherMap API.
3.  **Scoring**: Calculate a "Weather Score" for each city based on temperature, rain probability, wind, etc., to identify the best destinations.
4.  **Accommodation**: Scrape Booking.com for the top hotels in each city.
5.  **Storage**: Save the cleaned data into a **Cloudflare R2** bucket (data lake) and a **PostgreSQL database** (data warehouse) for further analysis or visualization.

> The forecast covers **5 days**, not the 7 mentioned in the assignment: OpenWeatherMap has since moved its 7-day endpoint out of the free plan. See the README for this and the other documented deviations.

In [1]:
# Import libraries
import pandas as pd
import requests
import json
import time
import datetime
import uuid
import os
import plotly.express as px
from dotenv import load_dotenv
import boto3
from sqlalchemy import create_engine, text, Table, Column, Integer, String, MetaData, ForeignKey, Float, Date

# Convenience only: nothing below may depend on a .env existing, exporting the
# same variables works just as well. Keys listed in .env.example.
load_dotenv()

# Configuration Variables
weather_api_secret = os.getenv('WEATHER_API_SECRET')

# Data lake -- Cloudflare R2, reached through the S3 API. The endpoint below is
# the only thing pointing boto3 away from AWS.
r2_access_key_id = os.getenv('R2_ACCESS_KEY_ID')
r2_secret_access_key = os.getenv('R2_SECRET_ACCESS_KEY')
r2_endpoint_url = os.getenv('R2_ENDPOINT_URL')
bucket_name = os.getenv('R2_BUCKET')

# Data warehouse -- managed PostgreSQL. Use the DIRECT endpoint, not the pooled
# one: the pooler runs in transaction mode and mishandles the DDL that
# meta.create_all() issues below.
host = os.getenv('HOST_SQL_ALCHEMY')
port = os.getenv('PORT_SQL_ALCHEMY')
database = os.getenv('DATABASE_SQL_ALCHEMY')
user = os.getenv('USER_SQL_ALCHEMY')
password = os.getenv('PASSWORD_SQL_ALCHEMY')

In [2]:
# Read initial list of cities from JSON
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

,city
0,Mont Saint Michel
1,St Malo
2,Bayeux
3,Le Havre
4,Rouen


In [3]:
# Create a working copy to avoid modifying the original dataframe
df = df_source.copy()

In [4]:
# Passed to both APIs below: a default python-requests user-agent gets refused.
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

# 1. Geolocation

In [ ]:
# Fetch coordinates for each city using Nominatim API
# We add a delay (time.sleep) to respect the API's usage policy and avoid rate limiting.

for index, row in df.iterrows():
    try:
        url = f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json"
        res = requests.get(url, headers=headers)
        
        if res.status_code == 200 and res.json():
            city_data = res.json()[0]
            df.loc[index, "lat"] = city_data["lat"]
            df.loc[index, "lon"] = city_data["lon"]
        else:
            print(f"Could not find coordinates for {row['city']}")
            
    except Exception as e:
        print(f"Error processing {row['city']}: {e}")
        
    time.sleep(1) # Pause to respect API rate limits

df.head()

# Checkpoint: Save result to CSV to avoid re-running expensive API calls
df.to_csv("cities_with_geoposition.csv", index=False)

In [5]:
# [Optional] Reload data from CSV if restarting the notebook
if os.path.exists("cities_with_geoposition.csv"):
    df = pd.read_csv("cities_with_geoposition.csv")
    print("Loaded cities data from CSV.")
else:
    print("CSV not found, using data from memory.")
    
df.head()

Loaded cities data from CSV.


,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966


# 2. Weather Forecast

In [ ]:
# Fetch 5-day/3-hour forecast data from OpenWeatherMap
list_weather_data = []

for index, row in df.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue

    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&appid={weather_api_secret}"
    res_weather = requests.get(url, headers=headers)
    res_weather_json = res_weather.json()
    
    if 'list' in res_weather_json:
        for res in res_weather_json['list']:
            weather_entry = {
                "city": row['city'],
                "lat": row['lat'],
                "lon": row['lon'],
                "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%Y-%m-%d'),
                "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
                "temp": res['main']['temp'],
                "prob_rain": res.get('pop', 0), # Probability of precipitation (0-1)
                # 'rain' is absent from the payload when no rain is expected
                "volume_rain": res.get('rain', {}).get('3h', 0),
                "wind_speed": res['wind']['speed'],
                "perc_cloud": res['clouds']['all']
            }
            list_weather_data.append(weather_entry)
    
    time.sleep(1) # Pause to respect API rate limits

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())

# Checkpoint: Save weather data
df_weather.to_csv("weather_forecast.csv", index=False)

In [6]:
# [Optional] Reload weather data
if os.path.exists("weather_forecast.csv"):
    df_weather = pd.read_csv("weather_forecast.csv")
    print("Loaded weather data from CSV.")
df_weather.head()

Loaded weather data from CSV.


,city,lat,lon,date,hour,temp,prob_rain,volume_rain,wind_speed,perc_cloud
0,Mont Saint Michel,48.635954,-1.51146,2026-08-16,20:00,21.34,0.0,0.0,5.91,90
1,Mont Saint Michel,48.635954,-1.51146,2026-08-16,23:00,20.45,0.0,0.0,4.71,90
2,Mont Saint Michel,48.635954,-1.51146,2026-08-17,02:00,18.86,0.0,0.0,1.84,93
3,Mont Saint Michel,48.635954,-1.51146,2026-08-17,05:00,17.08,0.0,0.0,2.62,100
4,Mont Saint Michel,48.635954,-1.51146,2026-08-17,08:00,18.01,0.0,0.0,3.93,99


In [7]:
# Data Pre-processing
# Convert probability of rain to percentage (0-100)
# Convert wind speed from m/s to km/h
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

,city,lat,lon,date,hour,temp,prob_rain,volume_rain,wind_speed,perc_cloud
0,Mont Saint Michel,48.635954,-1.51146,2026-08-16,20:00,21.34,0.0,0.0,21.276,90
1,Mont Saint Michel,48.635954,-1.51146,2026-08-16,23:00,20.45,0.0,0.0,16.956,90
2,Mont Saint Michel,48.635954,-1.51146,2026-08-17,02:00,18.86,0.0,0.0,6.624,93
3,Mont Saint Michel,48.635954,-1.51146,2026-08-17,05:00,17.08,0.0,0.0,9.432,100
4,Mont Saint Michel,48.635954,-1.51146,2026-08-17,08:00,18.01,0.0,0.0,14.148,99


In [8]:
# Aggregate the 3-hourly readings into one row per city and per day.
#
# Passing lists to .agg() gives pandas several columns per variable, which it
# can only name with a two-level column index: ('temp', 'max') and not
# 'temp_max'. That is why the scoring below indexes with tuples, and why the
# storage step has to flatten these names before writing to CSV and SQL.
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({
    'temp': ['mean', 'min', 'max'], 
    'prob_rain': 'max', 
    'volume_rain': ['mean', 'max', 'sum'], 
    'wind_speed': 'max', 
    'perc_cloud': 'mean'
}).reset_index()

df_weather_groupby.head()

city        lat      lon        date      temp                \
                                                      mean    min    max   
0  Aigues Mortes  43.566152  4.19154  2026-08-16  29.96000  28.94  30.98   
1  Aigues Mortes  43.566152  4.19154  2026-08-17  29.59000  23.16  36.02   
2  Aigues Mortes  43.566152  4.19154  2026-08-18  29.40875  24.49  33.47   
3  Aigues Mortes  43.566152  4.19154  2026-08-19  28.46000  25.92  32.61   
4  Aigues Mortes  43.566152  4.19154  2026-08-20  25.57750  22.14  28.28   

  prob_rain volume_rain             wind_speed perc_cloud  
        max        mean   max   sum        max       mean  
0      93.0     0.70500  0.87  1.41     18.180     65.000  
1      80.0     0.00000  0.00  0.00     30.816     64.250  
2       0.0     0.00000  0.00  0.00     21.348     50.500  
3      80.0     0.05375  0.43  0.43     39.744     37.125  
4     100.0     0.47000  1.28  3.76     22.968     60.125

## Scoring Methodology

We calculate a satisfaction score (0-100) for each weather metric, where **100 is perfect** and **0 is poor**.

### 1. Normalization (0-100)
| Metric | Target | Penalty Calculation |
| :--- | :--- | :--- |
| **Temperature** | 25°C | -4 pts per degree deviation from 25°C |
| **Rain Probability** | 0% | 100 - (Probability %) |
| **Rain Volume** | 0mm | -5 pts per mm |
| **Wind Speed** | 0 km/h | 100 - (Speed in km/h) |
| **Cloudiness** | 0% | 100 - (Cloud %) |

### 2. Weighted Final Score
The final score is a weighted average of individual scores:
- **Temperature**: 30%
- **Rain Probability**: 20%
- **Rain Volume**: 30% (Heavy penalty for rain)
- **Wind**: 10%
- **Clouds**: 10%

In [9]:
# 1. Calculate Individual Scores

# Temperature: Target 25°C
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Rain Probability: 0% is best
df_weather_groupby['score_rain_prob'] = 100 - (df_weather_groupby[('prob_rain', 'max')])

# Rain Volume: 0mm is best
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Wind Speed: 0 km/h is best
df_weather_groupby['score_wind'] = 100 - df_weather_groupby[('wind_speed', 'max')]
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Cloudiness: 0% is best
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Calculate Weighted Final Score
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. Rank the cities on their mean daily score over the whole forecast window,
# so that one perfect day does not outrank five merely good ones.
top_cities = df_weather_groupby.groupby(['city', 'lat', 'lon'])['total_score'].mean().sort_values(ascending=False)
print("--- FINAL RANKING ---")
print(top_cities.head(10))

df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)

--- FINAL RANKING ---
city                lat        lon      
La Rochelle         46.159732  -1.151595    87.888289
Bayonne             43.494514  -1.473666    85.748844
Paris               48.853495   2.348391    84.341022
Amiens              49.894171   2.295695    84.229061
Biarritz            43.483252  -1.559278    84.169406
Rouen               49.440459   1.093966    83.575761
Mont Saint Michel   48.635954  -1.511460    83.224533
St Malo             48.649518  -2.026041    83.164983
Bormes les Mimosas  43.150697   6.341928    83.098428
Cassis              43.214036   5.539632    81.750111
Name: total_score, dtype: float64


In [10]:
# Prepare DataFrame for visualization
# The warehouse keeps all 35 cities; only the map is narrowed to the Top-5 the
# deliverable asks for.
df_top_cities = pd.DataFrame(top_cities.reset_index())
df_top_cities_top_5 = df_top_cities.iloc[0:5, :]
df_top_cities_top_5

,city,lat,lon,total_score
0,La Rochelle,46.159732,-1.151595,87.888289
1,Bayonne,43.494514,-1.473666,85.748844
2,Paris,48.853495,2.348391,84.341022
3,Amiens,49.894171,2.295695,84.229061
4,Biarritz,43.483252,-1.559278,84.169406


In [18]:
# Map 1/2 of the deliverable: the Top-5 destinations by weather score
# scatter_map, not the deprecated scatter_mapbox: plotly moved from Mapbox to
# MapLibre, and the old wrapper warns on every call.
os.makedirs("images", exist_ok=True)

fig = px.scatter_map(
    df_top_cities_top_5, 
    lat="lat", 
    lon="lon",
    color="total_score",
    size="total_score", 
    color_continuous_scale=px.colors.sequential.Viridis,
    size_max=15,
    # Framed on the bounding box of all 35 candidate cities, not on the ones
    # plotted here: the frame must hold whichever five come out on top.
    zoom=5.4,
    center={"lat": 46.58, "lon": 2.86},
    map_style="carto-positron", 
    hover_name="city",
    # Display only: the stored score keeps full precision, because at one decimal
    # Amiens and Biarritz tie and sort order would decide ranks 4 and 5.
    hover_data={"total_score": ":.1f", "lat": False, "lon": False},
    title="Top 5 Destinations in France (Weather Based)"
)

# Also written to disk: GitHub renders none of plotly's interactive output.
# Portrait because France is taller than wide in Mercator, and set on the
# layout, not on write_image -- whose width/height would reach the PNG only,
# leaving fig.show() on a landscape default the zoom is not calibrated for.
fig.update_layout(width=1000, height=1100)

fig.write_image("images/top5_destinations.png", scale=2)
fig.show()

# 3. Accommodation (Booking.com)
We use a Scrapy spider to fetch hotel details for the top cities.

> **Run the spider from a terminal**, not from this notebook:
>
> ```bash
> cd booking_scraper_project
> scrapy crawl booking_spider -O ../hotels.json
> ```
>
> The crawl takes roughly an hour (`CONCURRENT_REQUESTS = 1`, `DOWNLOAD_DELAY = 2`).
> Scrapy logs to `logs/booking_spider_<timestamp>.log`. The cell below only
> checks that the resulting file is there.

In [11]:
# The crawl runs from a terminal, not from here: streaming an hour of Scrapy
# output into a single cell is what kills the notebook <-> kernel channel.
# See "Running it locally" in the README for the whole story.
#
#     cd booking_scraper_project
#     scrapy crawl booking_spider -O ../hotels.json
#
# Scrapy logs to logs/booking_spider_<timestamp>.log (see settings.py), and `-O`
# truncates hotels.json itself, so this cell deletes nothing. It only checks
# which crawl the analysis below is based on.

if not os.path.exists("hotels.json"):
    raise FileNotFoundError(
        "hotels.json is missing -- run the crawl in a terminal first "
        "(see the command above), then re-run this cell."
    )

_size_mb = os.path.getsize("hotels.json") / 1e6
_mtime = datetime.datetime.fromtimestamp(os.path.getmtime("hotels.json"))
print(f"Using hotels.json - {_size_mb:.1f} MB, crawled {_mtime:%Y-%m-%d %H:%M}")

Using hotels.json - 1.7 MB, crawled 2026-08-16 22:32


## Hotel Analysis & Visualization

In [12]:
# Load scraped hotel data
df_hotels = pd.read_json("hotels.json")

# Booking hides the review score of properties that have too few reviews; those
# rows cannot be ranked, so they are dropped rather than scored as zero.
df_hotels = df_hotels.dropna()

# Scores are scraped from a French page, where the decimal separator is a comma.
df_hotels['score'] = df_hotels['score'].astype(str).str.replace(',', '.').astype(float)
df_hotels = df_hotels.sort_values(by=["score"], ascending=False)

display(df_hotels.head())

,city,name,url,score,lat,lng,description
868,Bormes les Mimosas,SELECT'soHOME - Maison de Charme - Calme Absol...,https://www.booking.com/hotel/fr/select-sohome...,10.0,43.107553,6.354641,L’hébergement SELECT'soHOME - Maison de Charme...
823,Biarritz,Maison familiale cosy avec terrasse - Biarritz,https://www.booking.com/hotel/fr/maison-famili...,10.0,43.487398,-1.541602,"Situé à Biarritz, l’hébergement Maison familia..."
819,Biarritz,ELENA KEYWEEK Apartment sea view Biarritz down...,https://www.booking.com/hotel/fr/elena-keyweek...,10.0,43.481278,-1.566616,L’hébergement ELENA KEYWEEK Apartment sea view...
828,Biarritz,Villa Elisa - Maison esprit loft 500m plage,https://www.booking.com/hotel/fr/promo-elisa-a...,10.0,43.474086,-1.565243,L’hébergement Villa Elisa - Maison esprit loft...
699,Ariege,La Bulle d’Ax - Domaine de la Vallée dAx,https://www.booking.com/hotel/fr/la-bulle-d-ax...,10.0,42.714098,1.839371,L’hébergement La Bulle d’Ax - Domaine de la Va...


In [19]:
# Map 2/2 of the deliverable: the Top-20 hotels "in the area", i.e. ranked
# within the Top-5 destinations only, not nationally.
df_hotels_top_area = df_hotels[df_hotels['city'].isin(df_top_cities_top_5['city'])]

# df_hotels is already sorted by score, so head(20) is the Top-20 of that area.
fig = px.scatter_map(
    df_hotels_top_area.head(20), 
    lat="lat", 
    lon="lng",
    color="score",
    size="score", 
    color_continuous_scale=px.colors.sequential.Viridis,
    size_max=15,
    # Framed on the bounding box of all 35 candidate cities, not on the ones
    # plotted here: the frame must hold whichever five come out on top.
    zoom=5.4,
    center={"lat": 46.58, "lon": 2.86},
    map_style="carto-positron", 
    hover_name="name",
    # Display only; Booking already publishes one decimal. What this really
    # drops is the raw lat/lng plotly puts in the tooltip by default.
    hover_data={"score": ":.1f", "lat": False, "lng": False},
    title="Top 20 Hotels in the 5 Best Destinations"
)

# Also written to disk: GitHub renders none of plotly's interactive output.
# Portrait because France is taller than wide in Mercator, and set on the
# layout, not on write_image -- whose width/height would reach the PNG only,
# leaving fig.show() on a landscape default the zoom is not calibrated for.
fig.update_layout(width=1000, height=1100)

fig.write_image("images/top20_hotels.png", scale=2)
fig.show()

# 4. Data Storage (S3 & SQL)
Prepare the dataframes and upload them to the cloud for persistence.

In [13]:
# Data Formatting
# Assign UUIDs to cities and flatten the weather table for SQL compatibility

# 1. Add IDs to Cities
df_top_cities['id'] = [str(uuid.uuid4()) for _ in range(len(df_top_cities))]

# Save City Table
if not os.path.exists("final_output"):
    os.makedirs("final_output")
    
df_top_cities.rename(columns={'city': 'name'}).to_csv("final_output/villes_table.csv", index=False)

# 2. Flatten Weather Data (MultiIndex -> Single Level)
# .copy() and not a plain assignment: setting .columns on a reference would
# rename the aggregated DataFrame too.
df_weather_flat = df_weather_groupby.copy()
df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]

# Fixes the column list of the weather table, and drops lat/lon on the way:
# they belong to the cities table.
columns_weather = ['city', 'date', 'temp_mean', 'temp_min', 'temp_max', 
                   'prob_rain_max', 'volume_rain_mean', 'volume_rain_max', 'volume_rain_sum', 
                   'wind_speed_max', 'perc_cloud_mean', 'score_temp', 'score_rain_prob', 
                   'score_rain_vol', 'score_wind', 'score_cloud', 'total_score']
df_weather_flat = df_weather_flat[columns_weather]

# 3. Merge Function for ID linking
def merge_and_save(df_data, df_cities, filename):
    merged = df_data.merge(df_cities[['city', 'id']], on='city', how='left')
    merged = merged.rename(columns={'id': 'city_id'}).drop(columns=['city'])
    merged['id'] = [str(uuid.uuid4()) for _ in range(len(merged))]
    merged.to_csv(f"final_output/{filename}.csv", index=False)
    return merged

df_top_cities_light = df_top_cities[['id', 'city']]
df_weather_final = merge_and_save(df_weather_flat, df_top_cities_light, "weather_table")

df_hotel_final = merge_and_save(df_hotels, df_top_cities_light, "hotels_table")

In [14]:
# Load the cleaned CSVs into the data lake.
#
# Two arguments boto3 would otherwise infer from AWS have to be given by hand:
# the endpoint, which redirects the calls to R2, and a region -- R2 has none,
# but botocore signs every request with one, and "auto" is what R2 expects.
#
# No try/except on purpose: an upload that fails has to stop the ETL, not leave
# the lake out of sync with the warehouse behind a warning nobody reads.
s3 = boto3.client(
    "s3",
    endpoint_url=r2_endpoint_url,
    aws_access_key_id=r2_access_key_id,
    aws_secret_access_key=r2_secret_access_key,
    region_name="auto",
)

# The bucket is NOT created here: the API token is scoped to it with
# object-level rights only, so create_bucket() would fail by design.
files = [f for f in os.listdir("final_output") if f.endswith('.csv')]
for file in files:
    s3.upload_file(f"final_output/{file}", bucket_name, file)
    print(f"Uploaded {file} to R2.")

Uploaded weather_table.csv to R2.
Uploaded hotels_table.csv to R2.
Uploaded villes_table.csv to R2.


In [15]:
# Open the connection to the data warehouse.
#
# sslmode=require: the managed PostgreSQL refuses cleartext connections anyway,
# but psycopg2 defaults to "prefer", which silently falls back to plaintext if
# the handshake fails. Being explicit is what turns the encryption into a
# guarantee rather than a hope.
#
# The port is spelled out rather than left to the 5432 default, so that moving
# the warehouse only ever means editing .env.
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}?sslmode=require"
engine = create_engine(connection_string, echo=False)
meta = MetaData()

In [16]:
# 1. Cities Table
cities = Table(
    'cities', meta,
    Column('id', String, primary_key=True),
    Column('name', String),
    Column('lat', Float),
    Column('lon', Float),
    Column('total_score', Float)
)

# Idempotent: CREATE TABLE is only issued for what is missing, so re-running the
# notebook against a live warehouse is safe.
meta.create_all(engine)

# Refresh, not append: the primary keys are UUIDs regenerated on every run, so
# nothing would ever collide to stop the rows piling up. CASCADE reaches the two
# tables holding a foreign key to cities (no-op on a first run).
with engine.begin() as conn:
    conn.execute(text("TRUNCATE cities CASCADE"))

# The schema is owned by the Table() definitions above, not by pandas -- which
# would otherwise infer its own column types on the first write.
df_top_cities.rename(columns={'city': 'name'}).to_sql('cities', engine, if_exists='append', index=False)
print("Cities uploaded to SQL.")

Cities uploaded to SQL.


In [17]:
# 2. Weather Table
weather = Table(
    'weather', meta,
    Column('id', String, primary_key=True),
    Column('date', Date),
    Column('temp_mean', Float),
    Column('temp_min', Float),
    Column('temp_max', Float),
    Column('prob_rain_max', Float),
    Column('volume_rain_mean', Float),
    Column('volume_rain_max', Float),
    Column('volume_rain_sum', Float),
    Column('wind_speed_max', Float),
    Column('perc_cloud_mean', Float),
    Column('score_temp', Float),
    Column('score_rain_prob', Float),
    Column('score_rain_vol', Float),
    Column('score_wind', Float),
    Column('score_cloud', Float),
    Column('total_score', Float),
    Column('city_id', String, ForeignKey('cities.id'))
)

meta.create_all(engine)

# The CSV checkpoint reads dates back as plain strings; the column is a DATE.
df_weather_final['date'] = pd.to_datetime(df_weather_final['date'])
df_weather_final.to_sql('weather', engine, if_exists='append', index=False)
print("Weather uploaded to SQL.")

Weather uploaded to SQL.


In [18]:
# 3. Hotels Table
hotels = Table(
    'hotels', meta,
    Column('id', String, primary_key=True),
    Column('name', String),
    Column('url', String),
    Column('score', Float),
    Column('lat', Float),
    Column('lng', Float),
    Column('description', String),
    Column('city_id', String, ForeignKey('cities.id'))
)

meta.create_all(engine)
df_hotel_final.to_sql('hotels', engine, if_exists='append', index=False)
print("Hotels uploaded to SQL.")

Hotels uploaded to SQL.
